# LogiCrisis — GRPO Training on Colab
**Meta PyTorch OpenEnv Hackathon | Theme #1: Multi-Agent Interactions**

Stack: `Unsloth` 4-bit QLoRA + `TRL GRPOTrainer` + `LogiCrisisEnv`

> **Runtime**: GPU required — Runtime → Change runtime type → T4 GPU (free) or A100

## Step 1 — Install dependencies

In [ ]:
# Takes ~3 min on T4
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "trl>=0.9.0" datasets accelerate peft
!pip install -q fastapi uvicorn pydantic gradio httpx

## Step 2 — Load the LogiCrisis codebase

**Option A (recommended): Mount Google Drive**
1. Upload the `logicriasis/` folder to your Google Drive root
2. Run the cell below

**Option B: Clone from HuggingFace Spaces** (after you push)
```python
!git clone https://huggingface.co/spaces/YOUR_USERNAME/logicriasis
```

In [ ]:
# --- Option A: Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

import sys, os
LOGICRIASIS_PATH = '/content/drive/MyDrive/logicriasis'
sys.path.insert(0, LOGICRIASIS_PATH)
os.chdir(LOGICRIASIS_PATH)
print('Working directory:', os.getcwd())

# Verify import
from environment.tasks import ALL_TASK_IDS
print('Tasks loaded:', ALL_TASK_IDS)

## Step 3 — Heuristic baseline (before training)
Run this once and save the numbers. We'll compare after GRPO training.

In [ ]:
from inference import run_episode
from environment.tasks import ALL_TASK_IDS

print('=== HEURISTIC BASELINE (no LLM) ===')
baseline_scores = {}
for task_id in ALL_TASK_IDS:
    result = run_episode(task_id, use_llm=False)
    baseline_scores[task_id] = result['score']
    status = '✓ PASS' if result['passed'] else '✗ FAIL'
    print(f'  {task_id:<35} score={result["score"]:.4f}  {status}')

avg = sum(baseline_scores.values()) / len(baseline_scores)
print(f'\n  Average: {avg:.4f}')

## Step 4 — Load model with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/llama-3-8b-instruct-bnb-4bit"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,         # auto-detect (bfloat16 on Ampere+, float16 on T4)
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print('Model loaded:', MODEL_NAME)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Step 5 — Build prompt dataset from environment observations

In [ ]:
from training.train import build_prompt_dataset

# Start with Level 1 (easy) — curriculum learning
dataset = build_prompt_dataset(n_samples=128, curriculum_level=1)
print(f'Dataset: {len(dataset)} prompts')
print('\nSample prompt (first 500 chars):')
print(dataset[0]['prompt'][:500])

## Step 6 — Reward function sanity check

In [ ]:
from training.train import grpo_reward_fn, _score_completion

# Good: valid JSON action with reasoning
good = '{"action_type": "reroute", "cargo_id": "C001", "route_id": "Mumbai-Pune", "reasoning": "Rerouting around flood zone via open road"}'
# Medium: valid action, no reasoning
mid  = '{"action_type": "wait"}'
# Bad: malformed JSON
bad  = 'I will reroute the cargo to Mumbai'

print(f'Good action score : {_score_completion(good):+.3f}  (expect ~+0.6)')
print(f'Wait w/o reason   : {_score_completion(mid):+.3f}  (expect ~+0.2)')
print(f'Bad (no JSON) score: {_score_completion(bad):+.3f}  (expect -0.5)')

## Step 7 — GRPO Training

**Expected time on T4**: ~25–40 min for 128 samples × 1 epoch  
**Expected time on A100**: ~8–12 min

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.train import grpo_reward_fn

grpo_config = GRPOConfig(
    output_dir="/content/logicriasis_outputs",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    logging_steps=5,
    save_steps=50,
    report_to="none",
    max_completion_length=256,
    num_generations=4,       # 4 completions per prompt (use 8 on A100)
    temperature=0.7,
    seed=42,
    fp16=True,               # T4 uses fp16; A100 can use bf16
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[grpo_reward_fn],   # must be a list
    args=grpo_config,
    train_dataset=dataset,
)

print('Starting GRPO training…')
trainer.train()
print('Training complete!')

## Step 8 — Save LoRA adapters

In [ ]:
SAVE_DIR = "/content/logicriasis_outputs/final"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Adapters saved to {SAVE_DIR}')

# Optional: copy to Google Drive for persistence
import shutil
drive_save = '/content/drive/MyDrive/logicriasis_adapters'
shutil.copytree(SAVE_DIR, drive_save, dirs_exist_ok=True)
print(f'Also saved to Drive: {drive_save}')

## Step 9 — Before/After comparison
Compare heuristic baseline vs GRPO-trained LLM scores.

In [ ]:
import os
os.environ['API_BASE_URL'] = 'https://api-inference.huggingface.co/v1'
# os.environ['HF_TOKEN'] = 'hf_...'   # set your token if using HF inference API

# Run LLM inference using the fine-tuned model
# (plug trained model into run_episode by setting use_llm=True + env vars)
from inference import run_episode

print('=== GRPO-TRAINED LLM ===')
llm_scores = {}
for task_id in ALL_TASK_IDS:
    result = run_episode(task_id, use_llm=True)
    llm_scores[task_id] = result['score']
    status = '✓ PASS' if result['passed'] else '✗ FAIL'
    delta = result['score'] - baseline_scores.get(task_id, 0)
    sign = '+' if delta >= 0 else ''
    print(f'  {task_id:<35} score={result["score"]:.4f}  {status}  ({sign}{delta:.4f} vs heuristic)')

avg_llm = sum(llm_scores.values()) / len(llm_scores)
avg_base = sum(baseline_scores.values()) / len(baseline_scores)
print(f'\n  Average heuristic : {avg_base:.4f}')
print(f'  Average LLM (GRPO): {avg_llm:.4f}')
print(f'  Improvement       : {avg_llm - avg_base:+.4f}')

## Step 10 — Optional: Launch Gradio demo in Colab

In [ ]:
# Launch demo with public share link (valid for 72h)
import subprocess, threading

def run_demo():
    subprocess.run(['python', 'demo/app.py'])

t = threading.Thread(target=run_demo, daemon=True)
t.start()
print('Demo starting on port 7860 — check Gradio share link in output above')